In [1]:
import os
print(os.getcwd())

/Users/matt/Desktop/Patent Litigation Analysis


In [2]:
#importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re as re

In [4]:
pacer=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/pacer_cases_processed_v4.csv')
cases=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/cases_processed_v4.csv')
attorneys=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/attorneys_cleaned.csv')

In [5]:
attorneys.head(20)

,case_row_id,case_number,party_row_count,party_type,attorney_row_count,name,contactinfo,position,terminated_date,is_lead_attorney,is_inactive,firm,address,region
0,14,0:92-cv-00398-MJP,40,Plaintiff,1,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Collins and Lacy,"PO Box 12487, Columbia, SC 29211",South Carolina
1,14,0:92-cv-00398-MJP,41,Plaintiff,2,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Collins and Lacy,"PO Box 12487, Columbia, SC 29211",South Carolina
2,14,0:92-cv-00398-MJP,42,Plaintiff,3,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Collins and Lacy,"PO Box 12487, Columbia, SC 29211",South Carolina
3,14,0:92-cv-00398-MJP,43,Plaintiff,4,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Collins and Lacy,"PO Box 12487, Columbia, SC 29211",South Carolina
4,14,0:92-cv-00398-MJP,44,Plaintiff,5,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Collins and Lacy,"PO Box 12487, Columbia, SC 29211",South Carolina
5,14,0:92-cv-00398-MJP,45,Defendant,6,Edwin Russell Jeter,"Jeter and Williams; PO Box 7425; Columbia, SC ...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Jeter and Williams,"PO Box 7425, Columbia, SC 29202",South Carolina
6,14,0:92-cv-00398-MJP,45,Defendant,7,John Edward Cuttino,Gallivan White and Boyd; PO Box 7368; Columbia...,TERMINATED: 03/17/1994; LEAD ATTORNEY; ATTORNE...,1994-03-17,True,False,Gallivan White and Boyd,"PO Box 7368, Columbia, SC 29202",South Carolina
7,14,0:92-cv-00398-MJP,45,Defendant,8,Paul L Gardner,Spensley Horn Jubas and Lubitz; 1880 Century P...,LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaN,True,False,Spensley Horn Jubas and Lubitz,"1880 Century Park East, Suite 500, Los Angeles...",California
8,14,0:92-cv-00398-MJP,45,Defendant,9,R Bentz Kirby,Glenn Walters Law Firm; PO Box 1346; Orangebur...,TERMINATED: 06/05/1992; LEAD ATTORNEY; ATTORNE...,1992-06-05,True,False,Glenn Walters Law Firm,"PO Box 1346, Orangeburg, SC 29116",South Carolina
9,14,0:92-cv-00398-MJP,46,Counter Claimant,10,John Edward Cuttino,Gallivan White and Boyd; PO Box 7368; Columbia...,TERMINATED: 03/17/1994; LEAD ATTORNEY; ATTORNE...,1994-03-17,True,False,Gallivan White and Boyd,"PO Box 7368, Columbia, SC 29202",South Carolina


In [7]:
def standardize_case_number(case_num):
    if pd.isna(case_num):
        return case_num

    match = re.match(r'(\d+):(\d{2})-(\w+)-(\d+)(-\w+)?', case_num)

    if match:
        prefix = match.group(1)
        year = match.group(2)
        case_type = match.group(3).lower()
        number = match.group(4)

        year = '19' + year if int(year) >= 50 else '20' + year

        return f"{prefix}:{year}-{case_type}-{number}"

    return case_num

attorneys['case_number_clean'] = (
    attorneys['case_number']
    .apply(standardize_case_number)
)

In [9]:
attorneys['case_number_clean'].head(20)

0     0:1992-cv-00398
1     0:1992-cv-00398
2     0:1992-cv-00398
3     0:1992-cv-00398
4     0:1992-cv-00398
5     0:1992-cv-00398
6     0:1992-cv-00398
7     0:1992-cv-00398
8     0:1992-cv-00398
9     0:1992-cv-00398
10    0:1992-cv-00398
11    0:1992-cv-00398
12    0:1992-cv-00398
13    0:1992-cv-00398
14    0:1992-cv-00398
15    0:1992-cv-00398
16    0:1992-cv-00398
17    0:1992-cv-00398
18    0:1992-cv-00398
19    0:1992-cv-00398
Name: case_number_clean, dtype: str

In [11]:
cases['case_number_clean'] = cases['case_number_clean'].str.strip()
attorneys['case_number_clean'] = attorneys['case_number_clean'].str.strip()

# Get unique case numbers only
cases_unique = cases['case_number_clean'].dropna().drop_duplicates()
attorneys_unique = attorneys['case_number_clean'].dropna().drop_duplicates()

# Find which unique CASES case numbers are missing from ATTORNEYS
cases_not_in_attorneys = ~cases_unique.isin(attorneys_unique)

print("Unique case numbers in cases:", len(cases_unique))
print("Unique case numbers in attorneys:", len(attorneys_unique))
print("Unique case numbers in cases not found in attorneys:", cases_not_in_attorneys.sum())

Unique case numbers in cases: 69952
Unique case numbers in attorneys: 66702
Unique case numbers in cases not found in attorneys: 3809


In [14]:
cases['case_number_clean'] = cases['case_number_clean'].str.strip()
attorneys['case_number_clean'] = attorneys['case_number_clean'].str.strip()

# Create unique sets of case numbers
cases_unique = set(cases['case_number_clean'].dropna())
attorneys_unique = set(attorneys['case_number_clean'].dropna())

# Unique case numbers in CASES not found in ATTORNEYS
cases_only = cases_unique - attorneys_unique

# Unique case numbers in ATTORNEYS not found in CASES
attorneys_only = attorneys_unique - cases_unique

# Print counts
print("Unique case numbers only in cases:", len(cases_only))
print("Unique case numbers only in attorneys:", len(attorneys_only))

Unique case numbers only in cases: 3809
Unique case numbers only in attorneys: 559


In [ ]:
attorneys_unique.head(20)

AttributeError: 'set' object has no attribute 'head'